# Post Processing : Basics of FieldConvert utility

# Learning Objectives
Welocome to the tutorial on post-processing of Nektar++ simulations. This tutorial is aimed to show the basic usage of ***FieldConvert*** utitiliy which includes converting Nektar++ output files to appropriate format for post-processing software, primarily Paraview, VisIt or Tecplot as well as some basic manipulation of Nektar++ output files. More advanced usage of ***FieldConvert*** will be addressed in another tutorial.

-  Understanding the Nektar++ output files format
-  Learn how to convert the Nektar++ outputs for post-porcessing and visualization
-  Learn which post-processing software/programm can be used in conjugation with Nektar++
-  Learing the basics of the **FieldConvert** utility via several practical examples

# Prerequisite
There are not any prequisits for this tutorial. All the simulations files including mesh, session and outputs are provided. However, it might be helpful to have a look at incompressible Navier-Stokes or Advection-diffusion-Reaction tutorial (ADR) to get familiar with the common terminology and how to run the solver to obtain the simulation results.

# Overview

Nektar++ output files are in binary format and for visulalising the results we need to convert the outputs to a proper format.

Nektar++ output files may have either **.chk** or **.fld** extensions. **.chk** stands for **chekcpoints** and is used for the output files during the solution for example every n steps output during a transient simulation will have *.chk* extension. At the end of the simulaiton the final result is dumped with **.fld** extension which stands for **field** or **field data**. The two different extensions is just for distinguishing between checkpoint outputs and final results and do not have any other differences. 

Additionally, the outputs (both *chk* and *fld*) can be in **XML** or **HDF5** format (both in binary), where currently the xml is the default output. Inorder to output the files in HDF5 format one should build Nektar++ with HDF5 support and when running the solver uses the `-i Hdf5` command line option.

Nevertheless, no matter which format you are using or which extension your file has, any output from Nektar++ can be processed with **FieldConvert** utility.

Using the **FieldConvert** utility the simulations output can be converted to one of the following formats:
- **.vtu** or unstructured vtk format
- **.plt** Tecplot binary format
- **.dat** Tecplot Ascii format

The **.vtu** format can be opened using [ParaView](https://www.paraview.org) or [VisIt](https://wci.llnl.gov/simulation/computer-codes/visit), while the **plt** or **dat** formats are exclusive formats for [Tecplot](https://www.tecplot.com/). However, both [ParaView](https://www.paraview.org) and [VisIt](https://wci.llnl.gov/simulation/computer-codes/visit) are able to open the [Tecplot](https://www.tecplot.com/) formats so even you convert the file to for example plt, it is still possible to visualize the results using any of Paraview, VisIt or [Tecplot](https://www.tecplot.com/).

Besides converting the outputs to appropriate format for visulaization, **FieldConvert** can be used to modify, compute and extract various useful outputs and fields. For the details of FieldConvert capabilites pleaee consult   [Nektar++ user guide](https://www.nektar.info/getting-started/documentation/).


## Problem description
Lets assume we have simulated a '2D' incompressible flow over circular cylinder and now we want to visualise the result and process the output file to obtain `vorticity` field, `stream function` contours , etc. Further, lets assume that we had set up our simulaiton with separate [mesh](cylFlowMesh.xml) and [session](cylFlowSession.xml) files:

- Mesh file: [cylFlowMesh.xml](cylFlowMesh.xml)
- Session  file: [cylFlowSession.xml](cylFlowSession.xml)

The simulation is transient, starting from a [initial condition](init.rst) and continues for additional 15 seconds. The simulation writes the outputs every 100 steps ( every 0.5 sec ) producing checkpoints files *cylFlowMesh_100.chk* ... *cylFlowMesh_130.chk*. These checkpoints files are provided in the folder `transientResults`.

Additionally, at the end of simulaiton, an output file **cylFlowMesh.fld** is writtern.

We will use these files for the postprocessing and learning about the FieldConvert utility
`

# Basic File Conversion<a id='basic_conversion'></a>

The very basic format of using `FieldConvert` to convert the simulation result to for example ***.vtk*** format (for Paraview or VisIt) is

` FieldConvert meshFile.xml sessionFile.xml result.fld output.vtk `

Note that if you have both mesh and session file in a single file, e.g. `meshAndSession.xml` the above command simply becomes

` FieldConvert meshAndSession.xml result.fld output.vtk `

Here our mesh file is [cylFlowMesh.xml](cylFlowMesh.xml) and our simulation conditions, i.e the session file is [cylFlowSession.xml](cylFlowSession.xml) and the final simulation result file is [cylFlowMesh.fld](cylFlowMesh.fld), therefore we can convert the result by running the following command which produces the `finalOutput.vtu`

<div class="alert alert-info">

**Note:** `-f` is supplied to force output, otherwise the user would be prompted whether to overwrite an existing file.

</div>

In [ ]:
!FieldConvert -f cylFlowMesh.xml cylFlowSession.xml cylFlowMesh.fld finalOutput.vtu

## Visualisation in Paraview
Running the above command will convert nektar++ binary output format to **vtu** format (vtk unstructured). To visualise the result simply download the finaloutput.vtu file and open it with Paraview (or VisIt) using the following command

`paraview finalOutput.vtu`


Any other processed file in this tutorial can be similarily visualised with paraview. 
For more information on how to visualise and post-process the result using paraview see [paraview tutorial](https://www.paraview.org/tutorials/)


## Visualisation in vtk.js

The cells below show how visualisation of a VTU output file from `FieldConvert` can be viewed directly in the notebook. We use the converted file produced in previous step, i.e. the `finalOutput.vtu`.

For example to visulaise the *u* velocity contour, run the following code


In [ ]:
import pyvista as pv

# First read the VTK file
mesh = pv.read('finalOutput.vtu')

# Now create an itkwidget to visualise this in-browser.
pl = pv.PlotterITK()
pl.add_mesh(mesh, scalars=mesh['u'], smooth_shading=True)
pl.show(True)


<div class="alert alert-info">

**Note:** You can see contours of other variables such as `v` or `p` by simply changing the variable in the dropdown menu of the interactive plot, e.g. select "points: p" to see the pressure contour.

</div>

# Extracting vorticity field

`FieldConvert` utility can be used to modify the field data and compute new fields. These capablities are supported via various `modules` which allow the user to further manipulate the Nektar++ output files. 

To use the module we should pass the command line argument `-m module_name` to the field convert and we can export the result in *.fld* format for further manimpulations or directly convert it `.vtu` as follow
` FieldConvert -m module_name mesh.xml session.xml solution.fld intermediateOutput.fld `

or

` FieldConvert -m module_name mesh.xml session.xml solution.fld finalOutput.vtu`

For example to extract the vorticity you can use the `vorticity` module. run the following to extract the vorticity field

In [ ]:
!FieldConvert -f -m vorticity cylFlowMesh.xml cylFlowSession.xml cylFlowMesh.fld outputVorticity.vtu

Running the above command extract the vorticity field and sotre it (in addition to the main variables u,v and p) in the file `outputVorticity.vtu`. You can open this file with paraview or VisIt. 

Here, to visiualize the result in browser we use vtk.js as before. Run the following codes to see the  vorticity contour...

In [ ]:
import pyvista as pv

# First read the VTK file
mesh = pv.read('outputVorticity.vtu')

# Now create an itkwidget to visualise this in-browser.
pl = pv.PlotterITK()
pl.add_mesh(mesh, scalars=mesh['W_z'], smooth_shading=True)
pl.show(True)

# Extra tasks

## Adding new field
Lets try to compute the vorticty field and then computing the velocity magnitude and add it as a new field, convert the final file to vtu and visualize it.

### Task 1: Extract the vorticity field
As a firs step extract the vorticity field, but this time, do not convert the result to `vtu` format. Instead output the results to `fld` for further manipulation.

`Hint` the command would be 

`FieldConvert -f -m vorticity mesh.xml session.xml solution.fld intermediateOutput.fld`

Open a new cell in the notebook and excecute the above command. Remember to use the actual mesh ([cylFlowMesh.xml](cylFlowMesh.xml)), session ([cylFlowSession.xml](cylFlowSession.xml)) fand solution ([cylFlowMesh.fld](cylFlowMesh.fld)) file names. You can use the `vortOutput.fld` for the namee of the intermediate results or set another name of your choice.


### Task 2: Compute the velocity magnitude 
FieldConvert allows you to compute a new field using the module `fieldfromstring`. The general usage of this module is as follows:

`FieldConvert -m fieldfromstring:fieldstr="x+y+u":fieldname="result" mesh.xml session.xml solution.fld newOutput.fld`

where the `fieldstr="x+y+u` is the mandatory parameters describing the operation on the variables and `fieldname` is an optional parameter specifiying the name of new field. The results would be stored in `newOutput.fld`.

Since we want to calcuate the velocity magnitude the `fieldstr` parameters becomes: `fieldstr="sqrt(u*u+v*v)"`. Note that for a 3D simulation we should also include w component of velocity.

Lets name the new variable `velMag`

Now you can add the new field with:

`FieldConvert -m fieldfromstring:fieldstr="sqrt(u*u+v*v)":fieldname="velMag" cylFlowMesh.xml cylFlowSession.xml vortOutput.fld finalOutput.fld`

run the above command to compute a new output file `finalOutput.fld` that contains both vorticity and velcoity magnitude. Now convert this file to vtu format and visualize it using the python code inside the notebook or paraview in your ownsystem.

### Task 3: Alltogether

It is possible to combine all of these steps together:
`FieldConvert -m vorticity -m fieldfromstring:fieldstr="sqrt(u*u+v*v)":fieldname="velMag" cylFlowMesh.xml cylFlowSession.xml cylFlowMesh.fld finalOutput.vtu`

try it and visualize the result.


## Animating the solution

It is usefull and often required to process several files which are the intermediate outputs of a transient solution. In our example we have 31 `chk` outputs ranging from 100 to 130. to convert all of these with a single command use the following python code:

```python
for n in range(100,131):
    !FieldConvert -f cylFlowMesh.xml cylFlowSession.xml ./transientResults/cylFlowMesh_$n\.chk output_$n\.vtu
```

This will convert all 31 files to vtu format. 

<div class="alert alert-warning">

**Note:** Since the `chk` files are in `./transientResults` folder, we need to add the path to the command. However, since we didn't add any path for the outputs, the converted files will be written in the current folder.

</div>

### Taks1
Add the vorticity module to the above command and run the python code to extract the vorticity for all files and convert them to vtu

### Task2
use the following python strip to animate the outputs.

```python
import pyvista as pv
import time
import ipywidgets as widgets

# Here we read the output data. Note that in the FieldConvert command above,
# we set the output file names to be output_0.vtu, output_1.vtu ...
# the " 'output_%d.vtu' % i " will produce the file names.
#
# In the subsequent sections, you should replace the "output_" part by the 
# name you give for the outputs.
meshes = [ pv.read('output_%d.vtu' % i) for i in range(100,130) ]

plotter = pv.PlotterITK(notebook=True)
plotter.add_mesh(meshes[0], scalars='W_z')
viewer = plotter.show(False)

# we have 31 output files
nfiles = 31

i = 1
while True:
    viewer.geometries = meshes[i-1]
    i = (i+1) % nfiles
    time.sleep(0.25)
    
```

<div class="alert alert-warning">

**Note:** You will need to stop the cell's execution by pressing the stop button in the Jupyter interface to stop the animation.

</div>

# End of the tutorial